# Aula 04: Limpeza de Dados: Valores Ausentes e Outliers


## Objetivos de aprendizagem

Ao final desta aula você será capaz de:

1. **Diagnosticar** valores ausentes e duplicatas em um conjunto de dados.
2. **Entender** os mecanismos de ausência (MCAR, MAR, MNAR) e como eles influenciam a estratégia de tratamento.
3. **Aplicar** estratégias de tratamento de *missing*: remoção vs. imputação (média, mediana, moda, por grupo).
4. **Detectar** e **tratar** outliers usando IQR e z-score, sabendo quando removê-los, limitá-los ou mantê-los.

---

## Por que limpar dados?

Dados do mundo real raramente vêm prontos: sensores falham, formulários ficam em branco, sistemas duplicam registros e erros de digitação criam valores impossíveis. A **limpeza** transforma dados brutos em dados confiáveis. As duas patologias mais comuns são **valores ausentes** e **outliers**.

### Mecanismos de ausência (por que o dado faltou?)

Entender *por que* um valor está ausente orienta o tratamento correto:

| Mecanismo | Significado | Exemplo | Implicação |
|---|---|---|---|
| **MCAR** (*Missing Completely At Random*) | A ausência não depende de nada | Tubo de sangue quebrou no laboratório | Remover é seguro (mas perde dados) |
| **MAR** (*Missing At Random*) | Depende de **outras** variáveis observadas | Homens respondem menos sobre depressão | Imputar condicionado a outras colunas |
| **MNAR** (*Missing Not At Random*) | Depende do **próprio** valor ausente | Quem ganha muito omite a renda | Mais delicado; pode exigir modelagem |

> Nunca elimine linhas/valores "no automático". Primeiro entenda a origem da ausência; só então escolha entre remover ou imputar.


## 1. Diagnóstico: *missing* e duplicatas

Usaremos o dataset `penguins` (medidas físicas de pinguins de 3 espécies). Antes de tratar, precisamos **medir**: quantos valores faltam, em quais colunas e se há linhas repetidas.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset('penguins')
print('Shape:', df.shape)
print('\nMissing por coluna:')
print(df.isnull().sum())
print('\nLinhas duplicadas:', df.duplicated().sum())

In [ ]:
# Diagnóstico mais rico: percentual de missing por coluna e por linha + visualização

# Percentual de ausência por coluna
pct_col = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
print('% de valores ausentes por coluna:')
print(pct_col)

# Quantas LINHAS têm ao menos um valor ausente?
linhas_com_na = df.isnull().any(axis=1).sum()
print(f'\nLinhas com ao menos 1 ausente: {linhas_com_na} de {len(df)} '
      f'({linhas_com_na/len(df):.1%})')

# Mapa visual de valores ausentes (branco = ausente): útil para ver padrões
plt.figure(figsize=(8, 3))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Mapa de valores ausentes (claro = ausente)')
plt.tight_layout()
plt.show()

## 2. Tratamento de *missing*: remover vs. imputar

Há duas grandes rotas: **descartar** os dados ausentes ou **estimar** (imputar) valores plausíveis no lugar. A escolha depende de quantos dados faltam e do mecanismo de ausência.

Uma heurística inicial e segura:

- **Numéricas → mediana** (robusta a outliers, ao contrário da média).
- **Categóricas → moda** (categoria mais frequente).

Vamos comparar as duas estratégias e observar o impacto no tamanho e na completude do dataset.

### Panorama das estratégias de imputação

| Estratégia | Quando usar | Prós | Contras |
|---|---|---|---|
| **Remover linhas** (`dropna`) | Poucos ausentes e MCAR | simples | perde dados; pode enviesar |
| **Remover colunas** | Coluna com % altíssimo de missing | simples | perde uma variável inteira |
| **Média** | Numérica ~simétrica | rápida | sensível a outliers; reduz variância |
| **Mediana** | Numérica com outliers/assimetria | robusta | ignora relações entre variáveis |
| **Moda** | Categóricas | simples | reforça a categoria dominante |
| **Por grupo** (`groupby`) | Existe subgrupo relevante (ex.: espécie) | mais fiel à realidade | exige variável de grupo confiável |
| **KNN / Iterativa** | Relações fortes entre colunas | usa correlações | mais custoso; requer cuidado com leakage |

> **Sinalizador de ausência:** às vezes o fato de estar ausente é informativo. Nesses casos, crie uma coluna indicadora (`col_ausente = df['col'].isnull()`) antes de imputar.

Abaixo comparamos **remoção** vs. **imputação simples** (mediana nas numéricas, moda nas categóricas).


In [ ]:
# Estratégia A — REMOÇÃO: descarta qualquer linha com ao menos 1 ausente
df_drop = df.dropna()

In [ ]:
# Estratégia B — IMPUTAÇÃO: mediana nas numéricas, moda nas categóricas
df_imput = df.copy()
for col in df_imput.select_dtypes('number'):          # colunas numéricas
    df_imput[col] = df_imput[col].fillna(df_imput[col].median())
for col in df_imput.select_dtypes(exclude='number'):  # colunas categóricas
    df_imput[col] = df_imput[col].fillna(df_imput[col].mode()[0])

In [ ]:
# Comparação: quantas linhas sobraram e quantos ausentes restaram?
print('Original :', df.shape, '| ausentes:', df.isnull().sum().sum())
print('dropna   :', df_drop.shape, '| ausentes:', df_drop.isnull().sum().sum())
print('imputado :', df_imput.shape, '| ausentes:', df_imput.isnull().sum().sum())
print(f'\nA remoção descartou {len(df) - len(df_drop)} linhas '
      f'({(len(df)-len(df_drop))/len(df):.1%} do total).')

## 3. Outliers: O que são e por que importam

Um **outlier** é uma observação que se afasta muito do padrão dos demais dados. Ele pode ser:

- **Erro** (digitação, falha de sensor, unidade trocada) → normalmente corrigir/remover.
- **Evento raro, mas verdadeiro** (um cliente que gasta 100x a média) → geralmente **manter**, pois é informação real.

Por isso, **detectar não é o mesmo que remover**: sempre investigue a causa antes de agir.

### Dois métodos clássicos de detecção univariada

| Método | Regra | Quando é melhor |
|---|---|---|
| **IQR (boxplot)** | fora de \[Q1 − 1{,}5·IQR, Q3 + 1{,}5·IQR\] | distribuições assimétricas; robusto |
| **z-score** | \|z\| > 3, onde z = (x − média)/desvio | distribuições aproximadamente normais |

**Detalhe importante:** outliers podem estar escondidos por **agregação**. No `penguins`, olhar todas as espécies juntas (global) pode não mostrar nada, mas **dentro de cada espécie** surgem pontos discrepantes. Sempre pergunte: *outlier em relação a quê?*


### Detecção por IQR: global vs. por grupo

Aplicando a regra do IQR, fora de \[Q1 − 1{,}5·IQR, Q3 + 1{,}5·IQR\], coluna a coluna. `penguins` é um dataset relativamente limpo: no total (**global**) quase não há outliers univariados, mas avaliar **dentro de cada espécie** revela pontos que a visão global mascara. Isso ilustra por que a pergunta *"outlier em relação a quê?"* é essencial.

In [ ]:
def n_outliers_iqr(serie):
    Q1, Q3 = serie.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    return ((serie < Q1 - 1.5*IQR) | (serie > Q3 + 1.5*IQR)).sum()

print('coluna'.ljust(20), 'global', 'por espécie')
for col in df_imput.select_dtypes('number'):
    glob = n_outliers_iqr(df_imput[col])
    por_esp = df_imput.groupby('species')[col].apply(n_outliers_iqr).sum()
    print(col.ljust(20), str(glob).center(6), por_esp)

In [ ]:
# Visualizando outliers e comparando IQR vs. z-score em 'body_mass_g'
from scipy import stats

col = 'body_mass_g'
serie = df_imput[col]

# --- z-score: |z| > 3 ---
z = np.abs(stats.zscore(serie))
mask_z = z > 3
print(f'Outliers por z-score (|z|>3) em {col}: {mask_z.sum()}')

# --- IQR ---
Q1, Q3 = serie.quantile([0.25, 0.75])
IQR = Q3 - Q1
lim_inf, lim_sup = Q1 - 1.5*IQR, Q3 + 1.5*IQR
mask_iqr = (serie < lim_inf) | (serie > lim_sup)
print(f'Outliers por IQR em {col}: {mask_iqr.sum()}  '
      f'(limites: {lim_inf:.0f} a {lim_sup:.0f})')

# --- Boxplot por espécie: revela discrepâncias que a visão global esconde ---
plt.figure(figsize=(7, 4))
sns.boxplot(data=df_imput, x='species', y=col)
plt.title(f'{col} por espécie — pontos além dos "bigodes" são candidatos a outlier')
plt.tight_layout()
plt.show()

### O que fazer com um outlier?

Depois de detectar e **investigar a causa**, escolha uma ação:

| Ação | Descrição | Quando |
|---|---|---|
| **Corrigir** | Ajustar o valor errado (ex.: unidade trocada) | erro identificável |
| **Remover** | Excluir a linha | erro claro e sem correção possível |
| **Capar / *Winsorize*** | Limitar ao percentil (ex.: p1 e p99) | manter a linha sem deixar o valor distorcer |
| **Transformar** | Aplicar `log`/`sqrt` para comprimir a cauda | distribuição muito assimétrica |
| **Manter** | Não fazer nada | é um evento raro, porém real e relevante |

> Documente **toda** decisão: qual outlier, qual causa suspeita e qual ação. Reprodutibilidade é parte da qualidade.


## Exercício: Limpeza guiada e comparação antes/depois

Trabalhe sobre o dataset `penguins`:

1. **Imputação por grupo:** preencha `body_mass_g` ausente com a **mediana da própria espécie** (`groupby('species')`) em vez da mediana global. Compare os dois resultados.
2. **Duplicatas:** verifique e remova linhas duplicadas, se houver.
3. **Outliers por z-score:** detecte outliers de `body_mass_g` com |z| > 3 e compare a quantidade com a do método IQR acima.
4. **Antes e depois:** gere dois boxplots de `body_mass_g` por espécie lado a lado, um com os dados **originais** e outro **após a limpeza** e descreva em 2-3 linhas o que mudou.
5. **Documente** cada decisão de limpeza e sua justificativa (base para a tarefa de casa).

In [ ]:
from scipy import stats

# 1. Imputação por grupo (mediana por espécie)

# 2. Duplicatas

# 3. Outliers via z-score (|z| > 3)

# 4. Boxplots antes e depois (use plt.subplots(1, 2, sharey=True))

# Seu código:
